# Find content from Wikidata

We use wikidata to find temporal content that we can potentially create consistent temporal data from.
We use a range of queries and deduplicate at the end.

In [ ]:
from src.data_extraction.general_extractor import GeneralSPARQLExtractor
import os
import pandas as pd

extractor = GeneralSPARQLExtractor()
wikidata_results =  []
for file in os.listdir('./config'):
    if '.txt' not in file:
        continue
    print(f"processing: {file}")
    with open(f'./config/{file}', 'r') as f:
        query = f.read()
    result = extractor.execute_query(query)
    wikidata_results = wikidata_results + result

extraction_df = pd.DataFrame(wikidata_results)
# Drop duplicates based on wikipediaPage column
extraction_df = extraction_df.drop_duplicates(subset=['wikipediaPage'])

extraction_df.to_csv('./data/raw/wikipedia_pages.csv', index=False)





# Extract pages from wikipedia

In [ ]:
from src.data_extraction.wikipedia_extractor import WikipediaExtractor


OUTPUT_PARQUET_PATH = './data/raw/WikipediaContent.parquet'

df = pd.read_csv('./data/raw/wikipedia_pages.csv')

extractor = WikipediaExtractor()
results = []
for idx, row in df.iterrows():
    url = row['wikipediaPage'].strip()
    event = row['event']
    eventLabel = row['eventLabel']
    try:
        result = extractor.extract_wikipedia_content(url)
        results.append({
            'event': event,
            'eventLabel': eventLabel,
            'url': url,
            'success': result.get('success', False),
            'title': result.get('title', ''),
            'content': result.get('content', ''),
            'error': result.get('error', ''),
            'content_length': result.get('content_length', 0),
        })
    except Exception as e:
        results.append({
            'event': event,
            'eventLabel': eventLabel,
            'url': url,
            'success': False,
            'title': '',
            'content': '',
            'error': str(e),
            'content_length': 0,
        })
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1} rows...")
out_df = pd.DataFrame(results)
out_df.to_parquet(OUTPUT_PARQUET_PATH, index=False)
print(f"Saved {len(out_df)} rows to {OUTPUT_PARQUET_PATH}")


## Fill in the errors

In [ ]:
from src.data_extraction.general_extractor import GeneralSPARQLExtractor
import os
import pandas as pd
import re
from src.utils.wikipedia_extractor import WikipediaExtractor
INPUT_PARQUET_PATH = './data/raw/WikipediaContent.parquet'
OUTPUT_PARQUET_PATH = './data/raw/WikipediaContent_3.parquet'

wiki_df = pd.read_parquet(INPUT_PARQUET_PATH)

# Get URLs where success is False
failed_urls = wiki_df[~wiki_df['success']]['url'].tolist()

# Get records from df where wikipedia_url_id matches any of the failed URLs
failed_records = wiki_df[wiki_df['wikipediaPage'].isin(failed_urls)].reset_index()
print(f"Found {len(failed_records)} failed records")

extractor = WikipediaExtractor()
results = []
for idx, row in failed_records.iterrows():
    url = row['wikipediaPage'].strip()
    event = row['event']
    eventLabel = row['eventLabel']
    try:
        result = extractor.extract_wikipedia_content(url)
        results.append({
            'event': event,
            'eventLabel': eventLabel,
            'url': url,
            'success': result.get('success', False),
            'title': result.get('title', ''),
            'content': result.get('content', ''),
            'error': result.get('error', ''),
            'content_length': result.get('content_length', 0),
        })
    except Exception as e:
        results.append({
            'event': event,
            'eventLabel': eventLabel,
            'url': url,
            'success': False,
            'title': '',
            'content': '',
            'error': str(e),
            'content_length': 0,
        })
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1} rows...")
out_df = pd.DataFrame(results)
out_df.to_parquet(OUTPUT_PARQUET_PATH, index=False)


In [ ]:
### IF you need to add a range of parquets together
# Read all three parquet files
df1 = pd.read_parquet("data/raw/WikipediaContent.parquet")
# df2 = pd.read_parquet("data/raw/WikipediaContent_2.parquet") 
# df3 = pd.read_parquet("data/raw/WikipediaContent_3.parquet")

# Concatenate all dataframes
combined_df = pd.concat([df1], ignore_index=True)

# Drop duplicates based on url, keeping the first occurrence
combined_df = combined_df.drop_duplicates(subset=['url'], keep='first')

# Reset index
combined_df = combined_df.reset_index(drop=True)

# Save combined dataframe to parquet file
combined_df.to_parquet("./data/raw/WikipediaContent_combined.parquet", index=False)


print(f"Total rows after combining and deduplication: {len(combined_df)}")


# Parquet augmentation

Create an id column for all the new wikipedia entries

In [ ]:
import pandas as pd
import re

# Read the parquet file
df = pd.read_parquet("./data/raw/WikipediaContent_combined.parquet")

# Add an id column as integer index starting from 0
df['id'] = [str(i) for i in range(len(df))]

# Extract the last part of the URL after the last '/'
df['wikipedia_url_id'] = df['url'].str.split('/').str[-1]

# Function to clean string according to regex pattern
def clean_id(text):
    # Find all matches of alphanumeric, underscore and hyphen characters
    matches = re.findall(r'[a-zA-Z0-9_-]+', str(text))
    # Join matches and truncate to 64 chars if needed
    cleaned = ''.join(matches)[:64]
    return cleaned if cleaned else 'default'

# Clean the wikipedia_url_id column
df['wikipedia_url_id'] = df['wikipedia_url_id'].apply(clean_id)

print(f"Loaded {len(df)} rows")
print("\nFirst few rows:")
print(df[['id', 'wikipedia_url_id', 'url']].head())
# Save augmented dataframe to new parquet file
output_path = "./data/processed/WikipediaContent_augmented.parquet"
df.to_parquet(output_path)

print(f"\nSaved augmented dataframe to {output_path}")



In [ ]:
# Import env variables
# from dotenv import load_dotenv
import os

# Load environment variables
# load_dotenv()

# Get API key and model from environment variables
API_KEY = os.getenv('ANTHROPIC_API_KEY')
API_KEY



## Anthropic batching:

In [ ]:
# System and user prompts for temporal questions
from prompt_config.config import system_prompt, user_prompt
from src.utils.ai_question_generator import AIQuestionGenerator
from prompt_config.config import system_prompt, user_prompt
import os
import pandas as pd

# Example parquet file path (you'll need to adjust this to your actual file)
parquet_file = "data/processed/WikipediaContent_augmented.parquet"


generator = AIQuestionGenerator(provider="anthropic"
                                , api_key=API_KEY)

# Read the parquet file
df = pd.read_parquet(parquet_file)

# We specifically are looking for content that has relevance to us in 2024
df = df[df['content'].str.contains('2024', na=False)].reset_index()

# Prepare content for batch processing
batch_contents = []
for idx, row in df.iterrows():
    content_item = {
        'content': row['content'],
        'custom_id': row['id']  # Use URL as custom_id
    }
    batch_contents.append(content_item)


print(f"Creating batch with {len(batch_contents)} requests...")

# Create the batch
batch_info = generator.generate_qa_batch(
    contents=batch_contents,
    system_prompt=system_prompt,
    user_prompt=user_prompt,
    max_tokens=64000,
    num_questions=None
)

## OPTIONAL
You can test the question creator with this

In [ ]:
from src.utils.ai_question_generator import process_parquet_file
from prompt_config.config import system_prompt, user_prompt
import os


"""Example of basic parquet file processing."""
print("=== Basic Parquet Processing Example ===")

# Example parquet file path (you'll need to adjust this to your actual file)
parquet_file = "data/raw/WikipediaContent_augmented.parquet"

if not os.path.exists(parquet_file):
    print(f"Example file {parquet_file} not found. Please adjust the path to your actual parquet file.")
    sys.exit()

# Process the file with default settings
qa_pairs = process_parquet_file(
    parquet_file_path=parquet_file,
    system_prompt=system_prompt,
    user_prompt=user_prompt,
    provider="anthropic",  # or "openai" or "huggingface"
    api_key=API_KEY,
    model="claude-sonnet-4-20250514",
    num_questions_per_content=None, 
    max_tokens=64000,
    min_content_length=200  # Only process content with at least 200 characters
)
print(f"Generated {len(qa_pairs)} QA pairs")

# Save results to JSON
if qa_pairs:
    output_file = "data/processed/qa_pairs_basic.json"
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    
    import json
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(qa_pairs, f, indent=2, ensure_ascii=False)
    
    print(f"Results saved to: {output_file}")
